# **Garnet-OCR-3B-0422**

The [Garnet-OCR-3B-0422](https://huggingface.co/prithivMLmods/Garnet-OCR-3B-0422) model is a fine-tuned and optimized evolution of Megalodon-OCR-Sync-0713, built on top of the Qwen2.5-VL-3B-Instruct architecture. This version is specifically designed for high-precision mathematical formula extraction, structured markdown generation, and accurate table reconstruction, making it highly effective for technical, scientific, and structured documents. Trained on an enhanced mixture of document-centric datasets, including large-scale OCR-caption pairs and structured document corpora, the model improves layout fidelity, symbolic reasoning, and content structuring across diverse document types such as research papers, scanned PDFs, handwritten equations, and analytical reports.

In [ ]:
%%capture
!pip install git+https://github.com/huggingface/accelerate.git \
             git+https://github.com/huggingface/peft.git \
             transformers-stream-generator huggingface_hub albumentations \
             pyvips-binary qwen-vl-utils sentencepiece opencv-python docling-core \
             python-docx torchvision safetensors matplotlib num2words \

!pip install transformers==5.6.0 requests pymupdf hf_xet spaces pyvips pillow gradio==6.13.0 \
             einops torch fpdf timm av decord bitsandbytes fastapi
#Hold tight, this will take around 1-2 minutes.

### **Load & Try -> Garnet-OCR-3B-0422**

In [ ]:
import os
import sys
import time
import uuid
import json
import base64
import threading
from io import BytesIO
from pathlib import Path
from typing import Optional

import torch
import numpy as np
from PIL import Image

from gradio import Server
from fastapi import Request, UploadFile, File, Form
from fastapi.responses import HTMLResponse, JSONResponse, StreamingResponse

from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    TextIteratorStreamer,
    BitsAndBytesConfig,
)

# ── App Configuration ──────────────────────────────────────────────────────────
app = Server()

BASE_DIR  = Path(globals().get("__file__", "")).resolve().parent if globals().get("__file__") else Path.cwd()
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

MAX_MAX_NEW_TOKENS     = 4096
DEFAULT_MAX_NEW_TOKENS = 1024

# ── CUDA Info ──────────────────────────────────────────────────────────────────
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("torch.__version__    =", torch.__version__)
print("cuda available       :", torch.cuda.is_available())

if torch.cuda.is_available():
    DEVICE_LABEL = torch.cuda.get_device_name(torch.cuda.current_device()).lower()
else:
    DEVICE_LABEL = "cpu"

# ── Model Loading ──────────────────────────────────────────────────────────────
MODEL_ID = "prithivMLmods/Garnet-OCR-3B-0422"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_ID} with 4-bit quantization …")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=quantization_config,
    device_map="auto",
).eval()
print("Model ready.")


# ── Inference (SSE streaming) ──────────────────────────────────────────────────
def run_inference_stream(
    image: Image.Image,
    query: str,
    max_new_tokens: int,
    temperature: float,
    top_p: float,
    top_k: int,
    repetition_penalty: float,
):
    """Generator that yields SSE 'data: …\n\n' lines."""

    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": query},
        ],
    }]
    prompt_full = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = processor(
        text=[prompt_full],
        images=[image],
        return_tensors="pt",
        padding=True,
    ).to(model.device)

    streamer = TextIteratorStreamer(
        processor, skip_prompt=True, skip_special_tokens=True
    )
    generation_kwargs = {
        **inputs,
        "streamer": streamer,
        "max_new_tokens": max_new_tokens,
        "do_sample": True,
        "temperature": temperature,
        "top_p": top_p,
        "top_k": top_k,
        "repetition_penalty": repetition_penalty,
    }

    thread = threading.Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    buffer = ""
    for new_text in streamer:
        buffer += new_text
        buffer = buffer.replace("<|im_end|>", "")
        payload = json.dumps({"token": new_text.replace("<|im_end|>", ""), "full": buffer})
        yield f"data: {payload}\n\n"
        time.sleep(0.01)

    thread.join()
    yield "data: [DONE]\n\n"


# ── FastAPI Endpoints ──────────────────────────────────────────────────────────
@app.post("/api/ocr")
async def ocr_endpoint(
    query: str = Form("Extract all text from the image."),
    max_new_tokens: int = Form(DEFAULT_MAX_NEW_TOKENS),
    temperature: float = Form(0.7),
    top_p: float = Form(0.9),
    top_k: int = Form(50),
    repetition_penalty: float = Form(1.1),
    image: Optional[UploadFile] = File(None),
):
    if image is None or not image.filename:
        return JSONResponse({"error": "No image uploaded."}, status_code=400)

    content = await image.read()
    try:
        pil_img = Image.open(BytesIO(content)).convert("RGB")
    except Exception as e:
        return JSONResponse({"error": f"Invalid image: {e}"}, status_code=400)

    return StreamingResponse(
        run_inference_stream(
            pil_img, query,
            max_new_tokens, temperature, top_p, top_k, repetition_penalty,
        ),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "X-Accel-Buffering": "no",
        },
    )


# ── Frontend ───────────────────────────────────────────────────────────────────
@app.get("/", response_class=HTMLResponse)
async def homepage(request: Request):
    return f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8"/>
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>Garnet-OCR-3B</title>
  <link href="https://fonts.googleapis.com/css2?family=Syne:wght@400;600;700;800&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
  <style>
    /* ── Reset & Tokens ───────────────────────────────── */
    *, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}

    :root {{
      --bg:          #0d0d0f;
      --surface:     #141417;
      --surface-2:   #1c1c21;
      --border:      rgba(255,255,255,0.07);
      --border-hi:   rgba(255,255,255,0.14);
      --accent:      #d4a843;
      --accent-dim:  rgba(212,168,67,0.15);
      --accent-glow: rgba(212,168,67,0.35);
      --red:         #e05252;
      --green:       #52c97a;
      --blue:        #52a9e0;
      --text:        #f0ede8;
      --muted:       #7a7870;
      --mono:        'JetBrains Mono', monospace;
      --sans:        'Syne', sans-serif;
      --radius:      6px;
    }}

    body {{
      background: var(--bg);
      color: var(--text);
      font-family: var(--sans);
      min-height: 100vh;
      display: flex;
      flex-direction: column;
    }}

    /* ── Topbar ───────────────────────────────────────── */
    .topbar {{
      background: var(--surface);
      border-bottom: 1px solid var(--border);
      padding: 14px 28px;
      display: flex;
      align-items: center;
      justify-content: space-between;
      flex-shrink: 0;
    }}
    .topbar-brand {{
      display: flex; align-items: center; gap: 12px;
    }}
    .topbar-gem {{
      width: 10px; height: 10px;
      background: var(--accent);
      border-radius: 50%;
      box-shadow: 0 0 8px var(--accent-glow);
      flex-shrink: 0;
    }}
    .topbar-title {{
      font-size: 1.05rem; font-weight: 700; letter-spacing: 0.5px;
      color: var(--accent);
    }}
    .topbar-badge {{
      background: var(--accent-dim);
      border: 1px solid var(--accent);
      color: var(--accent);
      font-family: var(--mono);
      font-size: 11px;
      padding: 3px 10px;
      border-radius: 20px;
      letter-spacing: 0.5px;
    }}
    .topbar-device {{
      font-family: var(--mono);
      font-size: 11px;
      color: var(--muted);
      background: var(--surface-2);
      border: 1px solid var(--border);
      padding: 4px 12px;
      border-radius: 4px;
    }}

    /* ── Container ────────────────────────────────────── */
    .container {{
      max-width: 1380px;
      margin: 0 auto;
      padding: 32px 24px;
      flex: 1;
      width: 100%;
    }}

    .hero {{
      text-align: center;
      margin-bottom: 36px;
    }}
    .hero h1 {{
      font-size: 2.8rem;
      font-weight: 800;
      letter-spacing: -1px;
      color: var(--text);
      line-height: 1.1;
    }}
    .hero h1 span {{
      color: var(--accent);
    }}
    .hero p {{
      color: var(--muted);
      margin-top: 10px;
      font-size: 1rem;
      font-weight: 400;
    }}

    /* ── Layout Grid ──────────────────────────────────── */
    .layout {{
      display: grid;
      grid-template-columns: 400px 1fr;
      gap: 20px;
      min-height: 640px;
    }}

    /* ── Panel ────────────────────────────────────────── */
    .panel {{
      background: var(--surface);
      border: 1px solid var(--border);
      border-radius: var(--radius);
      display: flex;
      flex-direction: column;
      overflow: hidden;
    }}
    .panel-header {{
      padding: 14px 20px;
      background: var(--surface-2);
      border-bottom: 1px solid var(--border);
      font-size: 0.78rem;
      font-weight: 700;
      letter-spacing: 1.5px;
      text-transform: uppercase;
      color: var(--muted);
      display: flex;
      align-items: center;
      gap: 8px;
      flex-shrink: 0;
    }}
    .panel-header .dot {{
      width: 7px; height: 7px;
      border-radius: 50%;
      background: var(--accent);
      box-shadow: 0 0 6px var(--accent-glow);
    }}
    .panel-body {{
      flex: 1;
      padding: 20px;
      overflow-y: auto;
      display: flex;
      flex-direction: column;
      gap: 18px;
    }}

    /* ── Form Elements ────────────────────────────────── */
    label.field-label {{
      font-size: 12px;
      font-weight: 600;
      letter-spacing: 0.8px;
      text-transform: uppercase;
      color: var(--muted);
      margin-bottom: 6px;
      display: block;
    }}

    textarea, input[type="number"], input[type="text"] {{
      width: 100%;
      background: var(--surface-2);
      border: 1px solid var(--border);
      color: var(--text);
      font-family: var(--mono);
      font-size: 13px;
      padding: 10px 12px;
      border-radius: var(--radius);
      outline: none;
      transition: border-color 0.15s;
    }}
    textarea:focus, input:focus {{
      border-color: var(--accent);
    }}
    textarea {{
      min-height: 90px;
      resize: vertical;
      line-height: 1.5;
    }}

    /* ── Upload Zone ──────────────────────────────────── */
    .upload-zone {{
      background: var(--surface-2);
      border: 1.5px dashed var(--border-hi);
      border-radius: var(--radius);
      min-height: 210px;
      display: flex;
      flex-direction: column;
      align-items: center;
      justify-content: center;
      cursor: pointer;
      transition: border-color 0.2s, background 0.2s;
      position: relative;
      overflow: hidden;
    }}
    .upload-zone.dragover,
    .upload-zone:hover {{
      border-color: var(--accent);
      background: var(--accent-dim);
    }}
    .upload-zone input {{ display: none; }}
    .upload-icon {{
      width: 44px; height: 44px;
      border: 2px solid var(--border-hi);
      border-radius: 50%;
      display: flex; align-items: center; justify-content: center;
      margin-bottom: 10px;
      color: var(--muted);
      transition: border-color 0.2s, color 0.2s;
      pointer-events: none;
    }}
    .upload-zone:hover .upload-icon, .upload-zone.dragover .upload-icon {{
      border-color: var(--accent); color: var(--accent);
    }}
    .upload-hint {{
      font-size: 12px; color: var(--muted); text-align: center;
      pointer-events: none;
      line-height: 1.6;
    }}
    #previewImg {{
      display: none;
      width: 100%; height: 100%;
      object-fit: contain;
      position: absolute; inset: 0;
      border-radius: var(--radius);
    }}
    .clear-img {{
      display: none;
      position: absolute; top: 8px; right: 8px;
      background: rgba(0,0,0,0.7); color: white;
      border: none; border-radius: 50%;
      width: 26px; height: 26px;
      font-size: 14px; cursor: pointer;
      align-items: center; justify-content: center;
      z-index: 5;
    }}

    /* ── Advanced Accordion ───────────────────────────── */
    .adv-toggle {{
      width: 100%; background: none; border: none;
      color: var(--muted); font-family: var(--sans);
      font-size: 12px; font-weight: 700;
      letter-spacing: 1px; text-transform: uppercase;
      padding: 8px 0; cursor: pointer;
      display: flex; justify-content: space-between; align-items: center;
      transition: color 0.15s;
    }}
    .adv-toggle:hover {{ color: var(--accent); }}
    .adv-toggle .chevron {{ font-size: 16px; line-height: 1; }}
    .adv-body {{ display: none; }}
    .adv-body.open {{ display: block; padding-top: 14px; }}

    .grid-2 {{
      display: grid;
      grid-template-columns: 1fr 1fr;
      gap: 12px;
    }}
    .grid-2 .full {{ grid-column: span 2; }}

    /* Slider row */
    .slider-row {{
      display: flex; flex-direction: column; gap: 4px;
    }}
    .slider-row input[type="range"] {{
      -webkit-appearance: none;
      width: 100%; height: 4px;
      background: var(--surface-2);
      border-radius: 2px; border: none; outline: none;
      padding: 0;
    }}
    .slider-row input[type="range"]::-webkit-slider-thumb {{
      -webkit-appearance: none;
      width: 14px; height: 14px;
      background: var(--accent);
      border-radius: 50%;
      cursor: pointer;
      box-shadow: 0 0 6px var(--accent-glow);
    }}
    .slider-meta {{
      display: flex; justify-content: space-between;
      font-family: var(--mono); font-size: 11px; color: var(--muted);
    }}

    /* ── Run Button ───────────────────────────────────── */
    .btn-run {{
      width: 100%;
      padding: 14px;
      background: var(--accent);
      color: #0d0d0f;
      font-family: var(--sans);
      font-size: 15px; font-weight: 800;
      letter-spacing: 0.5px;
      border: none; border-radius: var(--radius);
      cursor: pointer;
      box-shadow: 0 4px 20px var(--accent-glow);
      transition: opacity 0.15s, transform 0.1s;
      flex-shrink: 0;
    }}
    .btn-run:hover {{ opacity: 0.9; transform: translateY(-1px); }}
    .btn-run:active {{ transform: translateY(0); }}
    .btn-run:disabled {{ opacity: 0.45; cursor: not-allowed; transform: none; }}

    /* ── Right Panel ──────────────────────────────────── */
    .output-panel {{
      display: flex;
      flex-direction: column;
      gap: 0;
    }}

    /* Execution log */
    .log-panel {{
      background: var(--surface);
      border: 1px solid var(--border);
      border-radius: var(--radius);
      overflow: hidden;
      flex-shrink: 0;
      margin-bottom: 16px;
    }}
    .log-header {{
      background: var(--surface-2);
      border-bottom: 1px solid var(--border);
      padding: 10px 16px;
      font-size: 11px; font-weight: 700;
      letter-spacing: 1.5px; text-transform: uppercase;
      color: var(--muted);
      display: flex; align-items: center; gap: 8px;
    }}
    .log-pulse {{
      width: 7px; height: 7px; border-radius: 50%;
      background: var(--muted);
      transition: background 0.3s, box-shadow 0.3s;
    }}
    .log-pulse.active {{
      background: var(--green);
      box-shadow: 0 0 8px var(--green);
      animation: blink 1s ease-in-out infinite;
    }}
    @keyframes blink {{ 0%,100%{{opacity:1}} 50%{{opacity:0.4}} }}

    .log-body {{
      font-family: var(--mono); font-size: 12px;
      padding: 10px 14px; max-height: 100px; overflow-y: auto;
      display: flex; flex-direction: column; gap: 3px;
    }}
    .log-line .lt {{ color: #555; margin-right: 8px; }}
    .log-info  {{ color: var(--blue); }}
    .log-ok    {{ color: var(--green); }}
    .log-err   {{ color: var(--red); }}

    /* Stream output */
    .stream-panel {{
      flex: 1;
      background: var(--surface);
      border: 1px solid var(--border);
      border-radius: var(--radius);
      display: flex; flex-direction: column;
      overflow: hidden;
      min-height: 300px;
    }}
    .stream-header {{
      background: var(--surface-2);
      border-bottom: 1px solid var(--border);
      padding: 12px 16px;
      display: flex; align-items: center; justify-content: space-between;
      flex-shrink: 0;
    }}
    .stream-header-left {{
      font-size: 11px; font-weight: 700;
      letter-spacing: 1.5px; text-transform: uppercase; color: var(--muted);
    }}
    .stream-actions {{
      display: flex; gap: 8px;
    }}
    .icon-btn {{
      background: none; border: 1px solid var(--border);
      color: var(--muted); border-radius: 4px;
      padding: 4px 10px; font-family: var(--mono);
      font-size: 11px; cursor: pointer;
      transition: color 0.15s, border-color 0.15s;
    }}
    .icon-btn:hover {{ color: var(--accent); border-color: var(--accent); }}

    .tab-bar {{
      display: flex; border-bottom: 1px solid var(--border);
      flex-shrink: 0;
    }}
    .tab {{
      padding: 10px 20px; font-size: 12px; font-weight: 700;
      letter-spacing: 0.5px; cursor: pointer;
      color: var(--muted); border-bottom: 2px solid transparent;
      transition: color 0.15s, border-color 0.15s;
    }}
    .tab.active {{ color: var(--accent); border-bottom-color: var(--accent); }}

    .tab-content {{ flex: 1; overflow-y: auto; display: none; }}
    .tab-content.active {{ display: block; }}

    #rawOutput {{
      width: 100%; height: 100%;
      background: transparent; border: none; outline: none;
      color: var(--text); font-family: var(--mono); font-size: 13px;
      padding: 16px; resize: none; line-height: 1.7;
    }}

    .md-content {{
      padding: 20px 24px;
      font-family: var(--sans);
      font-size: 14px; line-height: 1.75;
      color: var(--text);
    }}
    .md-content h1, .md-content h2, .md-content h3 {{
      color: var(--accent); margin: 16px 0 8px; font-weight: 700;
    }}
    .md-content pre {{
      background: var(--surface-2); border: 1px solid var(--border);
      padding: 12px; border-radius: 4px; overflow-x: auto;
      font-family: var(--mono); font-size: 12px;
    }}
    .md-content code {{ font-family: var(--mono); color: var(--accent); }}
    .md-content table {{
      border-collapse: collapse; width: 100%; margin: 12px 0;
    }}
    .md-content th, .md-content td {{
      border: 1px solid var(--border);
      padding: 8px 12px; text-align: left;
    }}
    .md-content th {{ background: var(--surface-2); color: var(--accent); }}

    /* ── Loading overlay ─────────────────────────────── */
    .loader-overlay {{
      display: none;
      position: absolute; inset: 0;
      background: rgba(13,13,15,0.75);
      backdrop-filter: blur(4px);
      align-items: center; justify-content: center;
      z-index: 20; border-radius: var(--radius);
      flex-direction: column; gap: 16px;
    }}
    .loader-ring {{
      width: 48px; height: 48px;
      border: 3px solid rgba(212,168,67,0.15);
      border-top-color: var(--accent);
      border-radius: 50%;
      animation: spin 0.9s linear infinite;
    }}
    .loader-label {{
      font-size: 13px; color: var(--text); font-weight: 600;
      letter-spacing: 0.5px;
      animation: pulse 1.4s ease-in-out infinite;
    }}
    @keyframes spin   {{ to {{ transform: rotate(360deg); }} }}
    @keyframes pulse  {{ 0%,100%{{opacity:1}} 50%{{opacity:0.45}} }}

    /* ── Empty state ─────────────────────────────────── */
    .empty-state {{
      display: flex; flex-direction: column;
      align-items: center; justify-content: center;
      height: 200px; gap: 12px;
      color: var(--muted); font-size: 13px;
    }}
    .empty-icon {{
      display: flex; align-items: center; justify-content: center;
      margin-bottom: 4px;
    }}

    /* ── Footer ──────────────────────────────────────── */
    footer {{
      text-align: center; padding: 24px;
      font-size: 11px; color: var(--muted);
      border-top: 1px solid var(--border);
      font-family: var(--mono);
    }}

    /* ── Responsive ──────────────────────────────────── */
    @media (max-width: 900px) {{
      .layout {{ grid-template-columns: 1fr; }}
      .hero h1 {{ font-size: 2rem; }}
    }}
  </style>
</head>
<body>

<!-- Topbar -->
<div class="topbar">
  <div class="topbar-brand">
    <div class="topbar-gem"></div>
    <span class="topbar-title">Garnet-OCR</span>
    <span class="topbar-badge">3B · 4-bit</span>
  </div>
  <div class="topbar-device" id="deviceLabel">{DEVICE_LABEL}</div>
</div>

<!-- Main -->
<div class="container">
  <div class="hero">
    <h1>Document <span>&amp; Image</span> Intelligence</h1>
    <p>Powered by Garnet-OCR-7B-0422 · Qwen2.5-VL · 4-bit Quantized</p>
  </div>

  <div class="layout">

    <!-- LEFT: Settings panel -->
    <div class="panel">
      <div class="panel-header"><span class="dot"></span>Input &amp; Settings</div>
      <div class="panel-body">

        <!-- Upload -->
        <div>
          <label class="field-label">Image Upload</label>
          <div class="upload-zone" id="dropZone">
            <input type="file" id="fileInput" accept="image/*"/>
            <img id="previewImg" alt="preview"/>
            <button class="clear-img" id="clearBtn" title="Remove image">×</button>
            <div id="uploadUi">
              <div class="upload-icon">
                <svg width="20" height="20" fill="none" stroke="currentColor" stroke-width="1.5" viewBox="0 0 24 24">
                  <path stroke-linecap="round" stroke-linejoin="round" d="M3 16.5v2.25A2.25 2.25 0 005.25 21h13.5A2.25 2.25 0 0021 18.75V16.5m-13.5-9L12 3m0 0l4.5 4.5M12 3v13.5"/>
                </svg>
              </div>
              <div class="upload-hint">Click or drag &amp; drop an image<br><span style="font-size:11px;opacity:.6;">PNG · JPG · WEBP · PDF screenshots</span></div>
            </div>
          </div>
        </div>

        <!-- Query -->
        <div>
          <label class="field-label">Query / Instruction</label>
          <textarea id="queryInput" placeholder="e.g. Extract all text from this document, or describe what you see…">Extract all text from the image.</textarea>
        </div>

        <!-- Advanced -->
        <div>
          <button class="adv-toggle" id="advToggle">
            <span>Advanced Options</span>
            <span class="chevron" id="advChevron">+</span>
          </button>
          <div class="adv-body" id="advBody">
            <div class="grid-2">
              <div class="full slider-row">
                <label class="field-label">Max New Tokens <span id="tokensVal">1024</span></label>
                <input type="range" id="maxTokens" min="64" max="4096" step="64" value="1024"
                       oninput="document.getElementById('tokensVal').textContent=this.value"/>
                <div class="slider-meta"><span>64</span><span>4096</span></div>
              </div>
              <div class="slider-row">
                <label class="field-label">Temperature <span id="tempVal">0.70</span></label>
                <input type="range" id="temperature" min="0.1" max="2.0" step="0.05" value="0.70"
                       oninput="document.getElementById('tempVal').textContent=parseFloat(this.value).toFixed(2)"/>
                <div class="slider-meta"><span>0.1</span><span>2.0</span></div>
              </div>
              <div class="slider-row">
                <label class="field-label">Top-p <span id="topPVal">0.90</span></label>
                <input type="range" id="topP" min="0.05" max="1.0" step="0.05" value="0.90"
                       oninput="document.getElementById('topPVal').textContent=parseFloat(this.value).toFixed(2)"/>
                <div class="slider-meta"><span>0.05</span><span>1.0</span></div>
              </div>
              <div class="slider-row">
                <label class="field-label">Top-k <span id="topKVal">50</span></label>
                <input type="range" id="topK" min="1" max="200" step="1" value="50"
                       oninput="document.getElementById('topKVal').textContent=this.value"/>
                <div class="slider-meta"><span>1</span><span>200</span></div>
              </div>
              <div class="full slider-row">
                <label class="field-label">Repetition Penalty <span id="repVal">1.10</span></label>
                <input type="range" id="repPenalty" min="1.0" max="2.0" step="0.05" value="1.10"
                       oninput="document.getElementById('repVal').textContent=parseFloat(this.value).toFixed(2)"/>
                <div class="slider-meta"><span>1.0</span><span>2.0</span></div>
              </div>
            </div>
          </div>
        </div>

        <!-- Run -->
        <button class="btn-run" id="runBtn">Run OCR</button>
      </div>
    </div>

    <!-- RIGHT: Output panel -->
    <div class="output-panel">

      <!-- Log -->
      <div class="log-panel">
        <div class="log-header">
          <div class="log-pulse" id="logPulse"></div>
          Execution Log
        </div>
        <div class="log-body" id="logBody">
          <div class="log-line"><span class="lt">[boot]</span><span class="log-ok">System ready · {DEVICE_LABEL}</span></div>
        </div>
      </div>

      <!-- Stream -->
      <div class="stream-panel" style="position:relative;">
        <div class="stream-header">
          <span class="stream-header-left">Model Output</span>
          <div class="stream-actions">
            <button class="icon-btn" id="copyBtn" title="Copy raw text">Copy</button>
            <button class="icon-btn" id="clearOutBtn" title="Clear output">Clear</button>
          </div>
        </div>

        <div class="tab-bar">
          <div class="tab active" data-tab="raw">Raw Stream</div>
          <div class="tab" data-tab="md">Rendered Markdown</div>
        </div>

        <div class="tab-content active" id="tab-raw">
          <textarea id="rawOutput" readonly placeholder="Output will stream here…"></textarea>
        </div>
        <div class="tab-content" id="tab-md">
          <div class="md-content" id="mdContent">
            <div class="empty-state">
              <div class="empty-icon">
                <svg width="32" height="32" fill="none" stroke="currentColor" stroke-width="1.2" viewBox="0 0 24 24" opacity="0.3">
                  <path stroke-linecap="round" stroke-linejoin="round" d="M19.5 14.25v-2.625a3.375 3.375 0 00-3.375-3.375h-1.5A1.125 1.125 0 0113.5 7.125v-1.5a3.375 3.375 0 00-3.375-3.375H8.25m0 12.75h7.5m-7.5 3H12M10.5 2.25H5.625c-.621 0-1.125.504-1.125 1.125v17.25c0 .621.504 1.125 1.125 1.125h12.75c.621 0 1.125-.504 1.125-1.125V11.25a9 9 0 00-9-9z"/>
                </svg>
              </div>
              <span>Rendered output will appear here after generation.</span>
            </div>
          </div>
        </div>

        <!-- Loader overlay -->
        <div class="loader-overlay" id="loaderOverlay">
          <div class="loader-ring"></div>
          <div class="loader-label">Streaming tokens…</div>
        </div>
      </div>

    </div>
  </div>
</div>

<footer>Garnet-OCR-3B-0422 · prithivMLmods · Qwen2.5-VL Architecture · 4-bit NF4 Quantization</footer>

<script>
// ── Minimal Markdown renderer ─────────────────────────────────────────────────
function renderMarkdown(text) {{
  if (!text) return '';
  let html = text
    // code blocks
    .replace(/```([\\s\\S]*?)```/g, '<pre><code>$1</code></pre>')
    // inline code
    .replace(/`([^`]+)`/g, '<code>$1</code>')
    // headings
    .replace(/^### (.+)$/gm, '<h3>$1</h3>')
    .replace(/^## (.+)$/gm, '<h2>$1</h2>')
    .replace(/^# (.+)$/gm, '<h1>$1</h1>')
    // bold / italic
    .replace(/\\*\\*(.+?)\\*\\*/g, '<strong>$1</strong>')
    .replace(/\\*(.+?)\\*/g, '<em>$1</em>')
    // tables (simple)
    .replace(/^(\\|.+\\|)$/gm, (m) => {{
      const cells = m.split('|').filter(c => c.trim() !== '');
      return '<tr>' + cells.map(c => `<td>${{c.trim()}}</td>`).join('') + '</tr>';
    }})
    // hr
    .replace(/^---+$/gm, '<hr/>')
    // newlines
    .replace(/\\n\\n/g, '</p><p>')
    .replace(/\\n/g, '<br/>');
  return `<p>${{html}}</p>`;
}}

// ── State ─────────────────────────────────────────────────────────────────────
let selectedFile = null;

// ── DOM refs ──────────────────────────────────────────────────────────────────
const dropZone    = document.getElementById('dropZone');
const fileInput   = document.getElementById('fileInput');
const previewImg  = document.getElementById('previewImg');
const clearBtn    = document.getElementById('clearBtn');
const uploadUi    = document.getElementById('uploadUi');
const queryInput  = document.getElementById('queryInput');
const runBtn      = document.getElementById('runBtn');
const logBody     = document.getElementById('logBody');
const logPulse    = document.getElementById('logPulse');
const rawOutput   = document.getElementById('rawOutput');
const mdContent   = document.getElementById('mdContent');
const loaderOverlay = document.getElementById('loaderOverlay');
const copyBtn     = document.getElementById('copyBtn');
const clearOutBtn = document.getElementById('clearOutBtn');

// ── Logging ───────────────────────────────────────────────────────────────────
function log(msg, cls='') {{
  const now = new Date().toLocaleTimeString('en-US', {{hour12:false}});
  const div = document.createElement('div');
  div.className = 'log-line';
  div.innerHTML = `<span class="lt">[${{now}}]</span><span class="${{cls}}">${{msg}}</span>`;
  logBody.appendChild(div);
  logBody.scrollTop = logBody.scrollHeight;
}}

// ── Advanced toggle ───────────────────────────────────────────────────────────
document.getElementById('advToggle').onclick = function() {{
  const body = document.getElementById('advBody');
  body.classList.toggle('open');
  document.getElementById('advChevron').textContent =
    body.classList.contains('open') ? '−' : '+';
}};

// ── Tabs ──────────────────────────────────────────────────────────────────────
document.querySelectorAll('.tab').forEach(tab => {{
  tab.onclick = () => {{
    document.querySelectorAll('.tab').forEach(t => t.classList.remove('active'));
    document.querySelectorAll('.tab-content').forEach(c => c.classList.remove('active'));
    tab.classList.add('active');
    document.getElementById('tab-' + tab.dataset.tab).classList.add('active');
  }};
}});

// ── File upload ───────────────────────────────────────────────────────────────
function setFile(file) {{
  selectedFile = file;
  const url = URL.createObjectURL(file);
  previewImg.src = url;
  previewImg.style.display = 'block';
  clearBtn.style.display = 'flex';
  uploadUi.style.display = 'none';
  log(`Image loaded: ${{file.name}}`, 'log-info');
}}

function clearFile() {{
  selectedFile = null;
  previewImg.src = '';
  previewImg.style.display = 'none';
  clearBtn.style.display = 'none';
  uploadUi.style.display = 'flex';
  fileInput.value = '';
}}

dropZone.onclick = (e) => {{
  if (e.target === clearBtn || e.target === previewImg) return;
  if (!selectedFile) fileInput.click();
}};
clearBtn.onclick = (e) => {{ e.stopPropagation(); clearFile(); }};
fileInput.onchange = (e) => {{ if (e.target.files[0]) setFile(e.target.files[0]); fileInput.value=''; }};
dropZone.ondragover  = (e) => {{ e.preventDefault(); dropZone.classList.add('dragover'); }};
dropZone.ondragleave = () => dropZone.classList.remove('dragover');
dropZone.ondrop = (e) => {{
  e.preventDefault(); dropZone.classList.remove('dragover');
  if (e.dataTransfer.files[0]) setFile(e.dataTransfer.files[0]);
}};

// ── Copy / Clear ──────────────────────────────────────────────────────────────
copyBtn.onclick = () => {{
  if (!rawOutput.value) return;
  navigator.clipboard.writeText(rawOutput.value)
    .then(() => {{ copyBtn.textContent = 'Copied!'; setTimeout(() => copyBtn.textContent='Copy', 1500); }});
}};
clearOutBtn.onclick = () => {{
  rawOutput.value = '';
  mdContent.innerHTML = '<div class="empty-state"><div class="empty-icon"><svg width="32" height="32" fill="none" stroke="currentColor" stroke-width="1.2" viewBox="0 0 24 24" opacity="0.3"><path stroke-linecap="round" stroke-linejoin="round" d="M19.5 14.25v-2.625a3.375 3.375 0 00-3.375-3.375h-1.5A1.125 1.125 0 0113.5 7.125v-1.5a3.375 3.375 0 00-3.375-3.375H8.25m0 12.75h7.5m-7.5 3H12M10.5 2.25H5.625c-.621 0-1.125.504-1.125 1.125v17.25c0 .621.504 1.125 1.125 1.125h12.75c.621 0 1.125-.504 1.125-1.125V11.25a9 9 0 00-9-9z"/></svg></div><span>Rendered output will appear here after generation.</span></div>';
}};

// ── Run ───────────────────────────────────────────────────────────────────────
runBtn.onclick = async () => {{
  const query = queryInput.value.trim();
  if (!query)          {{ return alert('Please enter a query.'); }}
  if (!selectedFile)   {{ return alert('Please upload an image.'); }}

  log('Preparing request…', 'log-info');
  loaderOverlay.style.display = 'flex';
  runBtn.disabled = true;
  logPulse.classList.add('active');
  rawOutput.value = '';
  mdContent.innerHTML = '';

  const fd = new FormData();
  fd.append('query',            query);
  fd.append('max_new_tokens',   document.getElementById('maxTokens').value);
  fd.append('temperature',      document.getElementById('temperature').value);
  fd.append('top_p',            document.getElementById('topP').value);
  fd.append('top_k',            document.getElementById('topK').value);
  fd.append('repetition_penalty', document.getElementById('repPenalty').value);
  fd.append('image',            selectedFile);

  log('Sending to model inference endpoint…', 'log-info');

  try {{
    const res = await fetch('/api/ocr', {{ method: 'POST', body: fd }});

    if (!res.ok) {{
      const err = await res.json();
      log('Server error: ' + (err.error || res.statusText), 'log-err');
      return;
    }}

    loaderOverlay.style.display = 'none'; // hide once stream starts

    const reader = res.body.getReader();
    const decoder = new TextDecoder();
    let buf = '';

    log('Streaming tokens…', 'log-ok');

    while (true) {{
      const {{ done, value }} = await reader.read();
      if (done) break;

      buf += decoder.decode(value, {{ stream: true }});
      const lines = buf.split('\\n');
      buf = lines.pop(); // keep incomplete last line

      for (const line of lines) {{
        if (!line.startsWith('data: ')) continue;
        const payload = line.slice(6).trim();
        if (payload === '[DONE]') {{
          log('Generation complete.', 'log-ok');
          mdContent.innerHTML = renderMarkdown(rawOutput.value);
          break;
        }}
        try {{
          const obj = JSON.parse(payload);
          rawOutput.value = obj.full;
          rawOutput.scrollTop = rawOutput.scrollHeight;
        }} catch (_) {{}}
      }}
    }}

  }} catch(e) {{
    log('Network error: ' + e.message, 'log-err');
    loaderOverlay.style.display = 'none';
  }} finally {{
    runBtn.disabled = false;
    logPulse.classList.remove('active');
    loaderOverlay.style.display = 'none';
  }}
}};
</script>
</body>
</html>
"""

# ── Launch ─────────────────────────────────────────────────────────────────────
app.launch()